### Caching

#### 1- Using Langgraph module

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

In [ ]:
# Enable cache
set_llm_cache(InMemoryCache())

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
response1 = llm.invoke("What is Parkinson's disease?")
response2 = llm.invoke("What is Parkinson's disease?")

- good for small project , but not for production

#### 2 - Using Redis

- pip install redis

Run Redis locally

- docker run -d --name redis -p 6379:6379 redis

##### 1- K-V Cache

In [ ]:
import redis

In [ ]:
redis_client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

- basic operations

In [ ]:
# SET
redis_client.set("name", "Tanmay")

# GET
value = redis_client.get("name")

print(value)

# DELETE
redis_client.delete("name")

- cache an expiration time:

In [ ]:
redis_client.setex(
    "name",
    300,           # 5 minutes
    "Tanmay"
)
# remove after 5 minutes

- redis in langgraph

retreival operation

In [ ]:
def retrieve_documents(query):
    print("Calling vector database...")
    return ["doc1", "doc2", "doc3"]

initialize redis

In [ ]:
redis_client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)


redis node

In [ ]:
def retrieve_node(state):

    query = state["query"]

    # Create deterministic cache key
    query_hash = hashlib.sha256(
        query.encode()
    ).hexdigest()

    cache_key = f"retrieval:{query_hash}"

    # -------------------------
    # CACHE HIT
    # -------------------------

    cached = redis_client.get(cache_key)

    if cached:
        print("CACHE HIT")

        return {
            "documents": json.loads(cached)
        }

    # -------------------------
    # CACHE MISS
    # -------------------------

    print("CACHE MISS")

    documents = retrieve_documents(query)

    # Save result for 10 minutes
    redis_client.setex(
        cache_key,
        600,
        json.dumps(documents)
    )

    return {
        "documents": documents
    }

##### 5 - Session Storage

Simple JSON

In [ ]:
session_id = "abc123"

In [ ]:
{
    "user_id": "42",
    "messages": [
        {
            "role": "user",
            "content": "What is Parkinson's disease?"
        },
        {
            "role": "assistant",
            "content": "Parkinson's disease is..."
        }
    ],
    "disease": "Parkinson's disease"
}

In [ ]:
import redis
import json

In [ ]:
redis_client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)


In [ ]:
def save_session(session_id, session_data):

    key = f"session:{session_id}"

    redis_client.setex(
        key,
        3600,                    # 1 hour
        json.dumps(session_data)
    )


In [ ]:
def get_session(session_id):

    key = f"session:{session_id}"

    data = redis_client.get(key)

    if data is None:
        return None

    return json.loads(data)

- o/p

In [ ]:
session = {
    "user_id": "42",
    "messages": [
        {
            "role": "user",
            "content": "What is Parkinson's disease?"
        }
    ]
}

save_session("abc123", session)

print(get_session("abc123"))

##### 2 - Langgraph

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import uuid

In [ ]:
app = FastAPI()

In [ ]:
class ChatRequest(BaseModel):
    message: str
    session_id: str | None = None


In [ ]:
@app.post("/chat")
def chat(request: ChatRequest):

    session_id = request.session_id

    if session_id is None:
        session_id = str(uuid.uuid4())

    result = graph.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": request.message
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": session_id
            }
        }
    )

    return {
        "session_id": session_id,
        "response": result
    }

##### 3 - Google ADK

docker run -d --name redis -p 6379:6379 redis

In [ ]:
import redis
import json
import hashlib

In [ ]:
redis_client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

In [ ]:
def create_cache_key(prompt):

    prompt_hash = hashlib.sha256(
        prompt.encode("utf-8")
    ).hexdigest()

    return f"llm:{prompt_hash}"


In [ ]:
def get_cached_response(prompt):

    key = create_cache_key(prompt)

    cached = redis_client.get(key)

    if cached:
        print("CACHE HIT")
        return json.loads(cached)

    print("CACHE MISS")
    return None


In [ ]:

def save_cached_response(
    prompt: str,
    response: str,
):

    key = create_cache_key(prompt)

    redis_client.setex(
        key,
        300,          # 5 minutes
        response,
    )

- ADK Agent

In [ ]:
from google.adk.agents import Agent

In [ ]:
root_agent = Agent(
    name="assistant",
    model="gemini-2.5-flash",
    instruction="You are a helpful assistant.",
)

- ADK Runtime / Session Management

In [ ]:
session_service = InMemorySessionService()

In [ ]:
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

In [ ]:
async def run_adk_agent(prompt: str) -> str:

    user_id = "user1"
    session_id = "session1"

    # Create session
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=user_id,
        session_id=session_id,
    )

    content = types.Content(
        role="user",
        parts=[
            types.Part(text=prompt)
        ],
    )

    final_response = ""

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content,
    ):

        if event.is_final_response():

            if event.content and event.content.parts:

                final_response = (
                    event.content.parts[0].text
                )

    return final_response


- Cached Agent Call

In [ ]:
async def call_agent(prompt: str) -> str:

    # 1. Redis GET

    cached = get_cached_response(prompt)

    if cached is not None:
        return cached

    # 2. ADK → Gemini

    response = await run_adk_agent(prompt)

    # 3. Redis SETEX

    save_cached_response(
        prompt,
        response,
    )

    return response



-  Testing

In [ ]:
async def main():

    prompt = "What is Parkinson's disease?"

    print("\n--- First request ---")

    response = await call_agent(prompt)

    print(response)

    print("\n--- Second request ---")

    response = await call_agent(prompt)

    print(response)


In [ ]:
if __name__ == "__main__":
    asyncio.run(main())